In [1]:
from sklearn.metrics import accuracy_score, roc_curve, confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
from natsort import natsorted
from tqdm import tqdm
import gc
gc.collect()
torch.cuda.empty_cache()

In [2]:
def metrics(tn, fp, fn, tp):
    """
    Calculate confusion matrix and various performance metrics.
    Args:
        tn (int): True Negatives
        fp (int): False Positives
        fn (int): False Negatives
        tp (int): True Positives
    Returns:
        cm_df (pd.DataFrame): Confusion matrix as a DataFrame.
        metric_df (pd.DataFrame): Performance metrics as a DataFrame.
    """
    tn = tn.astype(np.float32)
    fp = fp.astype(np.float32)
    fn = fn.astype(np.float32)
    tp = tp.astype(np.float32)

    cm_dict = {'predict\\actual':['Positive', 'Negative']
               ,'Positive':[tp, fn]
               ,'Negative':[fp, tn]}

    cm_df = pd.DataFrame(cm_dict)
    
    accuracy = round((tp + tn) / (tp + tn + fp + fn), 4) if (tp + tn + fp + fn) > 0 else 0  
    sensitivity = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0
    specificity = round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0
    ppv = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0
    npv = round(tn / (tn + fn), 4) if (tn + fn) > 0 else 0
    mcc = round((tp * tn - fp * fn) / np.sqrt(float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn)), 4) if (tp + fp) > 0 and (tp + fn) > 0 and (tn + fp) > 0 and (tn + fn) > 0 else 0

    metric_dict = {'metric':['Accuruacy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'MCC'],
                   'Value':[accuracy, sensitivity, specificity, ppv, npv, mcc]}

    metric_df = pd.DataFrame(metric_dict)

    return cm_df, metric_df

In [3]:
dataset_dir = r'D:\M143020071\MI\raw_data_result\dunwei\ch1\sr500_0.5_50_MI_win10s_step2s_2-7m/'
save_data_dir = r"D:\M143020071\MI\xgboost_results\dunwei\ECG_Founder_feature\sr500_0.5_50_MI_win10s_step2s_2-7m_ECG_signal/"
save_combine_data_dir = save_data_dir + "combine/"

In [4]:
for save_dir in [save_data_dir, save_combine_data_dir]:
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f'create directory: {save_dir}')
    else:
        print(f'directory already exists: {save_dir}')

directory already exists: D:\M143020071\MI\xgboost_results\dunwei\ECG_Founder_feature\sr500_0.5_50_MI_win10s_step2s_2-7m_ECG_signal/
directory already exists: D:\M143020071\MI\xgboost_results\dunwei\ECG_Founder_feature\sr500_0.5_50_MI_win10s_step2s_2-7m_ECG_signal/combine/


In [ ]:
all_data_dict = np.load(os.path.join(dataset_dir, 'all_data_dict.npz'), allow_pickle=True)
non_mi_per_windows_df = pd.read_csv(os.path.join(dataset_dir, 'non_mi_per_windows.csv'), dtype={'research_id': str})
mi_per_windows_df = pd.read_csv(os.path.join(dataset_dir, 'mi_per_windows.csv'), dtype={'research_id': str})
print(non_mi_per_windows_df.head())
print(mi_per_windows_df.head())
print(all_data_dict['0002'].shape)
print(all_data_dict['0425'].shape)
# feature_file_path = r"D:\M143020071\MI\xgboost_results\dunwei\ECG_Founder_feature\sr500_0.5_50_MI_win10s_step2s_2-7m_ECG_signal\ECG_Founder_deep_features.npy"

# # 2. 載入特徵資料
# # 根據資料格式：第0欄是ID，第1欄是Label，第2欄之後是深度特徵 (1024維)
# print(f"正在載入模型特徵檔: {os.path.basename(feature_file_path)}...")
# full_feature_data = np.load(feature_file_path, allow_pickle=True)

# # 3. 初始化容器以滿足後續變數需求
# all_data_dict = {}
# mi_ids = set()
# non_mi_ids = set()

# # 4. 依照 ID 拆解大型矩陣 (這步驟取代了遍歷資料夾的動作)
# # 取得矩陣中所有不重複的受試者 ID
# unique_ids = np.unique(full_feature_data[:, 0])

# print(f"開始解析特徵，共計 {len(unique_ids)} 位受試者...")

# for res_id in unique_ids:
#     # 建立 mask 找出該 ID 的所有列
#     mask = (full_feature_data[:, 0] == res_id)
    
#     # 取得該受試者的資料：包含 Label(第1欄) 與 特徵(第2欄之後)
#     # 轉換為 float32 以節省記憶體並確保運算正確
#     # 產出的矩陣形狀會是 (該ID的視窗數, 1 + 1024)
#     patient_data = full_feature_data[mask, 1:].astype(np.float32)
    
#     # 存入字典，Key 為字串格式的 ID
#     all_data_dict[str(res_id)] = patient_data
    
#     # 判斷是 MI 還是 Non-MI (看 patient_data 的第一欄 Label)
#     if patient_data[0, 0] == 1:
#         mi_ids.add(str(res_id))
#     else:
#         non_mi_ids.add(str(res_id))

# # 5. 建立後續程式碼需要的 DataFrame (滿足變數結構需求)
# mi_per_windows_df = pd.DataFrame({'research_id': list(mi_ids)})
# non_mi_per_windows_df = pd.DataFrame({'research_id': list(non_mi_ids)})

# # 6. 驗證讀取與轉換結果
# print(f"--- 讀取完成 ---")
# print(f"MI 受試者數量: {len(mi_per_windows_df)}")
# print(f"Non-MI 受試者數量: {len(non_mi_per_windows_df)}")

# if len(all_data_dict) > 0:
#     sample_id = list(all_data_dict.keys())[0]
#     print(f"範例 ID ({sample_id}) 資料形狀: {all_data_dict[sample_id].shape} (預期為: 視窗數, 1_Label + 1024_DeepFeatures)")

# # 7. 釋放大型原始矩陣記憶體
# del full_feature_data
# gc.collect()

正在載入模型特徵檔: ECG_Founder_deep_features.npy...
開始解析特徵，共計 400 位受試者...
--- 讀取完成 ---
MI 受試者數量: 200
Non-MI 受試者數量: 200
範例 ID (0002) 資料形狀: (146, 1025) (預期為: 視窗數, 1_Label + 1024_DeepFeatures)


0

In [6]:
hyper_params_dict = {
    'objective': 'binary:logistic','booster': 'gbtree', 'eval_metric': 'auc', 'learning_rate': 0.05, 'n_estimators': 500, 'max_depth': 3, 'min_child_weight': 1,
    'gamma': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.01, 'reg_lambda': 1, 'random_state': 42
}
hyper_params_df = pd.DataFrame({'hyper_param': list(hyper_params_dict.keys()), 'value': list(hyper_params_dict.values())})
print(hyper_params_df)
hyper_params_df.to_csv(os.path.join(dataset_dir, 'hyper_params.csv'), index=False)

         hyper_param            value
0          objective  binary:logistic
1            booster           gbtree
2        eval_metric              auc
3      learning_rate             0.05
4       n_estimators              500
5          max_depth                3
6   min_child_weight                1
7              gamma              0.1
8          subsample              0.8
9   colsample_bytree              0.8
10         reg_alpha             0.01
11        reg_lambda                1
12      random_state               42


In [7]:
gc.collect()
torch.cuda.empty_cache()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
GROUP_IDS = np.hstack([non_mi_per_windows_df['research_id'].to_numpy(), mi_per_windows_df['research_id'].to_numpy()])
GROUP_LABELS = np.concatenate([np.zeros(len(non_mi_per_windows_df['research_id'].to_numpy())), np.ones(len(mi_per_windows_df['research_id'].to_numpy()))])

total_train_tp = total_train_fn = total_train_fp = total_train_tn = 0
total_test_tp = total_test_fn = total_test_fp = total_test_tn = 0
total_valid_tp = total_valid_fn = total_valid_fp = total_valid_tn = 0

for fold, (train_idx, test_idx) in enumerate(skf.split(GROUP_IDS, GROUP_LABELS)):
    print(f'Fold {fold + 1} / {skf.n_splits}')
    train_list = []
    test_list = []
    train_ids = []
    test_ids = []

    train_group_ids, test_group_ids = GROUP_IDS[train_idx], GROUP_IDS[test_idx]
    train_group_labels, test_group_labels = GROUP_LABELS[train_idx], GROUP_LABELS[test_idx]

    for train_id, train_label in zip(train_group_ids, train_group_labels):
        if train_label == 0:
            data = all_data_dict[train_id][:, :].copy()
        elif train_label == 1:
            data = all_data_dict[train_id][:, :].copy()
        
        train_list.append(data)
        train_ids.extend([str(train_id)] * data.shape[0])
    
    for test_id, test_label in zip(test_group_ids, test_group_labels):
        if test_label == 0:
            data = all_data_dict[test_id][:, :].copy()  
        elif test_label == 1:
            data = all_data_dict[test_id][:, :].copy()
        
        test_list.append(data)
        test_ids.extend([str(test_id)] * data.shape[0])
    
            
    train = np.vstack(train_list)
    train_ids_df = pd.DataFrame({'research_id': train_ids})
    test = np.vstack(test_list)
    test_ids_df = pd.DataFrame({'research_id': test_ids})
    print(f'Train shape : {train.shape}, Test shape : {test.shape}')

    X_train, y_train = train[:, 1:], train[:, 0]
    X_test, y_test = test[:, 1:], test[:, 0]
    print(f'X_Train shape : {X_train.shape}, X_Test shape : {X_test.shape}')
    
    gc.collect()
    torch.cuda.empty_cache()

    clf = XGBClassifier(**hyper_params_dict)
    clf.fit(X_train, y_train)

    gc.collect()
    torch.cuda.empty_cache()

    yhat_train = clf.predict(X_train)
    yhat_train_proba = clf.predict_proba(X_train)[:, 1]

    gc.collect()
    torch.cuda.empty_cache()

    yhat_test = clf.predict(X_test)
    yhat_test_proba = clf.predict_proba(X_test)[:, 1]

    gc.collect()
    torch.cuda.empty_cache()

    y_train_df = pd.DataFrame(y_train, columns=['y_train'])
    yhat_train_df = pd.DataFrame(yhat_train, columns=['yhat_train'])
    yhat_test_df = pd.DataFrame(yhat_test, columns=['yhat_test'])
    y_test_df = pd.DataFrame(y_test, columns=['y_test'])
    yhat_train_proba_df = pd.DataFrame(yhat_train_proba, columns=['yhat_train_proba'])
    yhat_test_proba_df = pd.DataFrame(yhat_test_proba, columns=['yhat_test_proba'])
    y_train_label = pd.concat([train_ids_df.reset_index(drop=True), y_train_df.reset_index(drop=True), yhat_train_df], axis=1)
    y_test_label = pd.concat([test_ids_df.reset_index(drop=True), y_test_df.reset_index(drop=True), yhat_test_df], axis=1)
    yhat_train_probability = pd.concat([train_ids_df.reset_index(drop=True), yhat_train_proba_df], axis=1)
    yhat_test_probability = pd.concat([test_ids_df.reset_index(drop=True), yhat_test_proba_df], axis=1)

    y_train_label.to_csv(os.path.join(save_data_dir, f'y_train_label_{fold + 1}.csv'), index=False)
    y_test_label.to_csv(os.path.join(save_data_dir, f'y_test_label_{fold + 1}.csv'), index=False)
    yhat_train_probability.to_csv(os.path.join(save_data_dir, f'yhat_train_probability_{fold + 1}.csv'), index=False)
    yhat_test_probability.to_csv(os.path.join(save_data_dir, f'yhat_test_probability_{fold + 1}.csv'), index=False)

    train_tn, train_fp, train_fn, train_tp = confusion_matrix(y_train, yhat_train).ravel()
    total_train_tp += train_tp
    total_train_fn += train_fn
    total_train_fp += train_fp
    total_train_tn += train_tn

    test_tn, test_fp, test_fn, test_tp = confusion_matrix(y_test, yhat_test).ravel()
    total_test_tp += test_tp
    total_test_fn += test_fn
    total_test_fp += test_fp
    total_test_tn += test_tn

train_cm_df, train_metric_df = metrics(total_train_tn, total_train_fp, total_train_fn, total_train_tp)
test_cm_df, test_metric_df = metrics(total_test_tn, total_test_fp, total_test_fn, total_test_tp)
train_cm_df.to_csv(save_combine_data_dir + f"train_cm.csv", index=False)
train_metric_df.to_csv(save_combine_data_dir + f"train_metric.csv", index=False)
test_cm_df.to_csv(save_combine_data_dir + f"test_cm.csv", index=False)
test_metric_df.to_csv(save_combine_data_dir + f"test_metric.csv", index=False)

print('===========   TRAIN RESULTS   ===========')
print(f"confusion matrix of train :\n{train_cm_df.to_string(index=False)}\n")
print(f"metric of train :\n{train_metric_df.to_string(index=False)}\n")

print('===========   TEST RESULTS   ===========')
print(f"confusion matrix of test :\n{test_cm_df.to_string(index=False)}\n")
print(f"metric of test :\n{test_metric_df.to_string(index=False)}\n")


Fold 1 / 5
Train shape : (46720, 1025), Test shape : (11680, 1025)
X_Train shape : (46720, 1024), X_Test shape : (11680, 1024)
Fold 2 / 5
Train shape : (46720, 1025), Test shape : (11680, 1025)
X_Train shape : (46720, 1024), X_Test shape : (11680, 1024)
Fold 3 / 5
Train shape : (46720, 1025), Test shape : (11680, 1025)
X_Train shape : (46720, 1024), X_Test shape : (11680, 1024)
Fold 4 / 5
Train shape : (46720, 1025), Test shape : (11680, 1025)
X_Train shape : (46720, 1024), X_Test shape : (11680, 1024)
Fold 5 / 5
Train shape : (46720, 1025), Test shape : (11680, 1025)
X_Train shape : (46720, 1024), X_Test shape : (11680, 1024)
===========   TRAIN RESULTS   ===========
confusion matrix of train :
predict\actual  Positive  Negative
      Positive  114665.0     925.0
      Negative    2135.0  115875.0

metric of train :
     metric  Value
  Accuruacy 0.9869
Sensitivity 0.9817
Specificity 0.9921
        PPV 0.9920
        NPV 0.9819
        MCC 0.9739

===========   TEST RESULTS   ========

In [16]:
for fold, (train_idx, test_idx) in enumerate(skf.split(GROUP_IDS, GROUP_LABELS)):
    # 取得這一輪的 ID
    train_group_ids = GROUP_IDS[train_idx]
    test_group_ids = GROUP_IDS[test_idx]

    # --- 只針對第一批進行列印 ---
    if fold == 4:
        print(f"========== FOLD {fold + 1} TEST LIST ==========")
        print(f"測試集受試者 ID 總數: {len(test_group_ids)}")
        print(f"名單如下：\n{test_group_ids}")
        print("==============================================")
        
        # 如果想要直接轉成 List 方便複製
        # print(test_group_ids.tolist())
    
    # ... 後續原本的 train_list, test_list 處理邏輯 ...

========== FOLD 5 TEST LIST ==========
測試集受試者 ID 總數: 80
名單如下：
['0802' '0673' '0656' '0900' '0529' '0532' '0638' '0762' '0584' '0823'
 '0556' '0499' '0723' '0543' '0710' '0473' '0610' '0692' '0830' '0464'
 '0512' '0885' '0801' '0894' '0432' '0655' '0515' '0740' '0683' '0633'
 '0463' '0449' '0510' '0608' '0578' '0614' '0611' '0711' '0588' '0866'
 '0104' '0328' '0209' '0271' '0086' '0336' '0235' '0290' '0197' '0152'
 '0404' '0210' '0112' '0029' '0251' '0108' '0082' '0113' '0159' '0013'
 '0237' '0021' '0256' '0358' '0302' '0383' '0264' '0303' '0270' '0010'
 '0369' '0243' '0346' '0312' '0135' '0226' '0002' '0047' '0332' '0372']
